# Cases: how a ranking goes wrong

[The tour](tour_from_a_model_to_a_ranking.ipynb) walks the package once, from a YAML model to a ranking. This notebook stops at four places on that path where the rule a reasonable person would write first gives a different answer, and runs that rule beside the package's. It ends on one thing the package cannot settle.

Read it at either depth. The first sentence of each case and its figure or table are enough to follow the argument. The cell, its `assert` lines and the **Source** line are there for the reader who wants to check it.

The four sites sit near several boundaries at once, so that each case shows its effect. The notebook shows that each effect can happen. It does not show how often.

Every case has the same parts.

1. **Question**, in one sentence.
2. **Fixture**: the committed numbers the case runs over. All are invented.
3. **Naive**: the obvious rule, computed in the cell.
4. **The repo's rule**: the same question through the package.
5. An `assert` that sets the two answers side by side.
6. **Source**: where the rule comes from, marked `Incident:` (the wrong answer happened here), `Design:` (a choice made here) or `Literature:` (a failure known outside), with the test that holds the rule.

The data has three levels, and section 1 shows all three before any case begins: what **arrives** (raw measurements, each in its own unit), what is **done to it** (a value function turns each into a utility from 0 to 1), and what is **reported** (a weighted total and a ranking).

The suite runs every cell of this notebook (`tests/unit/test_notebooks.py`) and fails a case that lacks a part, never compares its naive answer with the package's, or cites a test, file or commit that does not exist (`tests/unit/test_case_notebooks.py`).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent  # the notebook runs from notebooks/
sys.path.insert(0, str(ROOT))

In [ ]:
import itertools
import math
import warnings

import numpy as np
import yaml

from src.myproject.decision_science import (
    MAUTScorer,
    dominance_check,
    exponential,
    monte_carlo,
    passage_chart,
    robustness_report,
    slope_chart,
)

FIXTURES = ROOT / "tests" / "fixtures" / "tour"
assert (FIXTURES / "depot_model.yaml").exists(), "run this notebook from the notebooks/ folder of the repo"

scorer = MAUTScorer.from_yaml(FIXTURES / "depot_model.yaml")
sites = yaml.safe_load((FIXTURES / "depot_sites.yaml").read_text())["alternatives"]
WEIGHT = {c.name: c.weight for c in scorer.criteria}
SOURCE = "tests/fixtures/tour, invented numbers"


def table(header: list, rows: list[list]) -> None:
    """Print rows under a header, right-aligned; floats to three places, integers with separators."""
    def text(v):
        return f"{v:.3f}" if isinstance(v, float) else f"{v:,}" if isinstance(v, int) else str(v)
    cells = [[text(v) for v in row] for row in [header, *rows]]
    widths = [max(len(row[i]) for row in cells) for i in range(len(header))]
    for row in cells:
        print("  ".join(value.rjust(width) for value, width in zip(row, widths)))


def ranked(alternatives: dict) -> dict[str, float]:
    """The package's ranking as {site: utility}, best first, with the narrow-range warning set aside (case 4 reads it)."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return {r.alternative: r.utility for r in scorer.rank(alternatives)}


def best_first(scores: dict[str, float]) -> dict[str, float]:
    return dict(sorted(scores.items(), key=lambda item: -item[1]))

## 1. The passage, once: what arrives, what is done to it, what is reported

**What arrives.** Four candidate sites for a supply depot, each measured on four criteria in four different units: thousand dollars a year, minutes, pallets, and years without a flood. The first table prints the file whole.

**What is done to it.** The model gives each criterion a value function and a weight. The figure draws all of it: one panel per criterion, the raw measurement across, the utility up, each site a point on the curve. Two axes run backwards, cost and drive time, because less is better there; in every panel better is up and to the right. A panel's width is its weight.

**What is reported.** The last panel: each site's total, the sum of weight times utility. The second table is the same result as a ranking.

Follow River Bend. It is the best site on the two widest panels and near the floor on flood years, and it finishes second.

In [ ]:
table(["site", *WEIGHT], [[name, *(raw[c] for c in WEIGHT)] for name, raw in sites.items()])

results = scorer.rank(sites)
passage = passage_chart(
    scorer, sites, title="Four depot sites, from raw measurement to utility to total",
    source=SOURCE, highlight=["Airport Road", "River Bend"],
)

print()
table(["place", "site", "total", *WEIGHT],
      [[place, r.alternative, r.utility, *(r.breakdown[c] for c in WEIGHT)] for place, r in enumerate(results, start=1)])

repo = {r.alternative: r.utility for r in results}
assert list(repo) == ["Airport Road", "River Bend", "Rail Yard", "Old Mill"]
assert all(abs(sum(r.breakdown.values()) - r.utility) < 1e-9 for r in results), "a total is the sum of its parts"
assert len(passage.axes) == len(WEIGHT) + 1

## Case 1. A raw number is not a utility

**Question.** Which site is best?

**Fixture.** `depot_sites.yaml` and the weights in `depot_model.yaml`.

**Naive.** A weighted sum of the measurements as they arrive. It is what a spreadsheet's `SUMPRODUCT` over the weights row gives.

**The repo's rule.** No raw number is ever added to another. `MAUTScorer.score` passes each through its criterion's value function first, and refuses any result outside 0 to 1.

**Source.** Design: the additive form in `src/myproject/decision_science/scorer.py`. `test_the_weight_applies_to_the_utility_and_not_to_the_raw_number` in `tests/unit/test_scorer.py` holds the sum (it fails when a raw number is added in place of a utility), and `test_bad_value_function_raises` holds the refusal.

In [ ]:
naive_sum = best_first({name: sum(WEIGHT[c] * raw[c] for c in WEIGHT) for name, raw in sites.items()})
pallet_share = {name: WEIGHT["storage_pallets"] * sites[name]["storage_pallets"] / naive_sum[name] for name in sites}

table(["rule", "1st", "2nd", "3rd", "4th"],
      [["weighted sum of raw numbers", *naive_sum], ["the package", *repo]])
print()
table(["site", "naive total", "of which pallets", "the package"],
      [[name, naive_sum[name], f"{pallet_share[name]:.0%}", repo[name]] for name in naive_sum])

naive_leader, leader = next(iter(naive_sum)), next(iter(repo))
assert naive_leader == "Rail Yard" and leader == "Airport Road" and naive_leader != leader
assert min(pallet_share.values()) > 2 / 3, "one criterion with a 0.20 weight supplies over two thirds of every naive total"
assert list(naive_sum)[-1] == "River Bend" and min(sites, key=lambda s: sites[s]["annual_cost"]) == "River Bend", (
    "the cheapest site comes last, because the sum counts a dollar spent as a point earned"
)

The naive total is a count of pallets with some noise added: pallets run to the thousands, so a 0.20 weight on them outweighs everything else. It also rewards cost. Neither fault raises an error, and the naive table looks like a ranking.

## Case 2. The scale comes from the model, not from who turned up

**Question.** If the last-placed site withdraws, does the order of the others change?

**Fixture.** The same four sites, then the three left when Old Mill withdraws. Old Mill is dominated: Airport Road matches or beats it on every criterion, so it cannot win under any weights.

**Naive.** Fix case 1 the usual way: rescale each criterion from 0 to 1 between the worst and best site present, flip the two costs, then take the weighted sum. This is min-max normalization, and it needs no model beyond the weights.

**The repo's rule.** The scale is written in the model file, as `low` and `high` or a curve's own parameters, before any site is measured. A site's utility depends on that site alone.

**Source.** Design: `MAUTScorer.from_yaml` binds each value function's parameters from the model, and `score` takes one alternative at a time. `test_a_utility_does_not_depend_on_which_other_alternatives_are_ranked` in `tests/unit/test_scorer.py` holds it. The decision literature calls the naive result rank reversal; this notebook cites no paper for it.

In [ ]:
COSTS = ("annual_cost", "drive_minutes")


def naive_minmax(alternatives: dict) -> dict[str, float]:
    """Weighted sum after scaling each criterion between the worst and best alternative present."""
    totals = dict.fromkeys(alternatives, 0.0)
    for c, weight in WEIGHT.items():
        low, high = min(a[c] for a in alternatives.values()), max(a[c] for a in alternatives.values())
        for name, raw in alternatives.items():
            scaled = (raw[c] - low) / (high - low)
            totals[name] += weight * (1 - scaled if c in COSTS else scaled)
    return best_first(totals)


remaining = {name: raw for name, raw in sites.items() if name != "Old Mill"}
naive_four, naive_three = naive_minmax(sites), naive_minmax(remaining)
repo_three = ranked(remaining)

withdrawal = slope_chart(
    {"Naive: scaled to the sites present": {"four sites": naive_four, "Old Mill withdrawn": naive_three},
     "The package: scaled by the model": {"four sites": repo, "Old Mill withdrawn": repo_three}},
    title="A dominated site withdraws: the naive leader changes, and the package's ranking does not move",
    source=f"Score from 0 to 1, both panels on one scale. {SOURCE}.",
    highlight=["Airport Road", "River Bend", "Rail Yard"],
)

table(["site", "naive, four sites", "naive, Old Mill withdrawn", "package, four sites", "package, Old Mill withdrawn"],
      [[name, naive_four[name], naive_three.get(name, "withdrawn"), repo[name], repo_three.get(name, "withdrawn")] for name in repo])

assert len(withdrawal.axes) == 2
assert ("Airport Road", "Old Mill") in dominance_check(results), "the site that withdraws could never have won"
naive_leaders = (next(iter(naive_four)), next(iter(naive_three)))
assert naive_leaders == ("Rail Yard", "River Bend"), "the naive leader changes when a loser leaves"
assert list(naive_three)[-1] == "Airport Road" == next(iter(repo)), "and the package's leader is last of the three"
assert all(repo_three[name] == repo[name] for name in repo_three), "no utility moves in the package"
assert list(repo_three) == [name for name in repo if name != "Old Mill"] and naive_leaders[1] != next(iter(repo_three))
worst_on = [c for c in WEIGHT if sites["Old Mill"][c] == (max if c in COSTS else min)(raw[c] for raw in sites.values())]
assert worst_on == ["drive_minutes"], "Old Mill is last on drive time and on nothing else"
drive = {"Airport Road": sites["Airport Road"]["drive_minutes"]}
assert sorted(raw["drive_minutes"] for raw in sites.values()).index(drive["Airport Road"]) == 2, "third best of four on drive time"
assert max(raw["drive_minutes"] for raw in remaining.values()) == drive["Airport Road"], "and worst of three"
assert round(naive_four["Airport Road"] - naive_three["Airport Road"], 3) == 0.130, "Airport Road's naive total falls although nothing about it changed"
assert round(naive_four["Rail Yard"] - naive_four["River Bend"], 3) == 0.005, "the naive lead that changes hands is small"

Old Mill set the worst value on drive time and on nothing else. When it left, the drive-time scale shrank to fit the three that remained, and Airport Road's 35 minutes went from third best of four to worst of three. Airport Road's naive total fell from 0.547 to 0.417 although nothing about Airport Road had changed, and the naive leader changed from Rail Yard to River Bend. That lead was 0.005, so the fall in Airport Road's total is the larger effect. Under the naive rule a decision can be changed by adding or removing an option nobody would choose. The package's totals did not move.

The package pays for this with a requirement: someone must state the scale before the data arrives. That is the judgment the model file records.

## Case 3. A rate near zero is not zero

**Question.** What is the utility of a 50-minute drive on a scale from 90 down to 10, as the exponential curve flattens toward a straight line?

**Fixture.** One point, the middle of the range, and nine rates from 1e-6 down to 1e-17. A straight line gives exactly 0.5.

**Naive.** The formula as the textbook prints it: `(1 - exp(-rate * t)) / (1 - exp(-rate))`.

**The repo's rule.** Below 1e-8 in magnitude, `exponential` returns the straight-line value, which is the formula's own limit.

**Source.** Incident: the crash was found while a property-test strategy was being designed and filed as a Known Issue in `docs/doctrine-updates.md`: "Found during property-test strategy design, 2026-07-20." Commit `acdab6e` fixed it later and records a second fault, a silent wrong answer. Its message reads: "A rate under about 2.2e-16 raised ZeroDivisionError, and at 1e-12 the denominator's cancellation returned a wrong utility without raising." The message's 2.2e-16 is not where the crash starts; the table below measures it. `test_tiny_nonzero_rate_approaches_linear` in `tests/unit/test_value_functions.py` holds the fix.

In [ ]:
def naive_exponential(x: float, low: float, high: float, rate: float) -> float | str:
    t = (x - low) / (high - low)
    try:
        return (1 - math.exp(-rate * t)) / (1 - math.exp(-rate))
    except ZeroDivisionError as error:
        return type(error).__name__


RATES = (1e-6, 1e-9, 1e-12, 1e-15, 3e-16, 1.2e-16, 1e-16, 5e-17, 1e-17)
naive_at = {rate: naive_exponential(50, 90, 10, rate) for rate in RATES}
repo_at = {rate: exponential(50, low=90, high=10, rate=rate) for rate in RATES}

table(["rate", "textbook formula", "the package", "textbook error"],
      [[f"{rate:g}", f"{naive_at[rate]:.10f}" if isinstance(naive_at[rate], float) else naive_at[rate], f"{repo_at[rate]:.10f}",
        f"{naive_at[rate] - repo_at[rate]:+.2e}" if isinstance(naive_at[rate], float) else "no answer"] for rate in RATES])

assert naive_at[1e-6] == repo_at[1e-6], "above the threshold the package is the textbook formula"
assert abs(naive_at[1e-15] - repo_at[1e-15]) > 0.05, "at 1e-15 the textbook form is off by more than a twentieth of the scale"
assert round(naive_at[1e-15], 3) == 0.556, "a quiet wrong answer at 1e-15"
assert [round(naive_at[rate], 3) for rate in (3e-16, 1.2e-16, 1e-16)] == [0.333, 1.0, 0.0], "and 0.333, 1.000 and 0.000 below it, with no error"
assert naive_at[5e-17] == "ZeroDivisionError" == naive_at[1e-17] and repo_at[5e-17] == repo_at[1e-17] == 0.5
assert naive_exponential(50, 90, 10, 5.5e-17) == "ZeroDivisionError" and naive_exponential(50, 90, 10, 5.6e-17) == 0.0, (
    "the crash begins at 5.5e-17, not at the 2.2e-16 the commit message gives"
)
assert all(abs(value - 0.5) < 1e-6 for value in repo_at.values())

The textbook form fails in two ways, and the quiet one reaches further. At 1e-15 it returns 0.556 where the answer is 0.500, with no error: the denominator has lost its digits to cancellation. At 3e-16, 1.2e-16 and 1e-16 it returns 0.333, 1.000 and 0.000, also with no error. It stops with an error only at 5e-17 and below. Measured here, the crash begins at 5.5e-17 (5.5e-17 raises, 5.6e-17 returns 0.0), a quarter of the commit message's 2.2e-16.

## Case 4. A weight is not an influence

**Question.** Drive time has a weight of 0.30. Does it decide 30% of the choice among the three sites left after case 2?

**Fixture.** Rail Yard, Airport Road and River Bend. Their drive times are 22, 35 and 18 minutes.

**Naive.** Read the weight as the share: 0.30 of the decision.

**The repo's rule.** A criterion can separate the alternatives by no more than its weight times the range of utility it gives them. `rank` warns when that range is under 0.2, since the criterion's weight is then largely unspent.

**Source.** Design: the warning in `MAUTScorer.rank`, in `src/myproject/decision_science/scorer.py`. `test_warns_when_utility_range_is_narrow` in `tests/unit/test_scorer.py` holds the warning.

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    three = scorer.rank(remaining)

utility_range = {c: max(r.raw_utilities[c] for r in three) - min(r.raw_utilities[c] for r in three) for c in WEIGHT}
can_separate = {c: WEIGHT[c] * utility_range[c] for c in WEIGHT}
share = {c: can_separate[c] / sum(can_separate.values()) for c in WEIGHT}
naive_share = {c: WEIGHT[c] / sum(WEIGHT.values()) for c in WEIGHT}  # the weight, read as the share of the choice

table(["criterion", "weight", "naive share", "utility range", "can separate by", "share of what separates them"],
      [[c, f"{WEIGHT[c]:.2f}", f"{naive_share[c]:.0%}", utility_range[c], can_separate[c], f"{share[c]:.0%}"] for c in WEIGHT])
print()
for warning in caught:
    print("the package warns:", warning.message)

assert abs(naive_share["drive_minutes"] - 0.30) < 1e-9 and share["drive_minutes"] < naive_share["drive_minutes"] / 4, (
    "a 0.30 weight that carries under a tenth of the difference"
)
assert share["flood_free_years"] > share["drive_minutes"] + share["storage_pallets"], "flooding outweighs drive time and pallets together"
assert 5 < can_separate["flood_free_years"] / can_separate["drive_minutes"] < 6
assert [str(w.message).split("'")[1] for w in caught] == ["drive_minutes"], "the package names the criterion"
model = yaml.safe_load((FIXTURES / "depot_model.yaml").read_text())["criteria"]
scale = next(c for c in model if c["name"] == "drive_minutes")["params"]
assert (scale["low"], scale["high"]) == (90, 10), "the weight prices the swing from 90 minutes to 10"
assert WEIGHT["drive_minutes"] == max(WEIGHT.values()) and abs(WEIGHT["flood_free_years"] / WEIGHT["drive_minutes"] - 2 / 3) < 1e-9
assert round(utility_range["drive_minutes"], 3) == 0.101 and round(can_separate["drive_minutes"], 3) == 0.030

by_site = {r.alternative: r for r in three}
river_lead = by_site["River Bend"].utility - by_site["Rail Yard"].utility
from_drive = WEIGHT["drive_minutes"] * (by_site["River Bend"].raw_utilities["drive_minutes"] - by_site["Rail Yard"].raw_utilities["drive_minutes"])
assert round(river_lead, 3) == 0.004 and round(from_drive, 3) == 0.006 and from_drive > river_lead, (
    "drive time spends little of its weight and still supplies more than River Bend's whole lead over Rail Yard"
)

All three sites are a short drive, so the criterion with the joint-largest weight tells them apart by 0.030 out of a possible 0.300. The weight on flooding is two thirds as large and does nearly six times the work.

A weight is the value of the whole swing across a criterion's stated scale: 0.30 prices moving drive time from 90 minutes to 10. These three sites use about a tenth of that swing (a utility range of 0.101), so most of the weight goes unspent. This is the case for reading the warning and not silencing it. It marks a number that is correct and should not be believed as stated: the weights say what each full swing is worth, and they say little about what separates these three sites.

A small spread can still settle a close pair. Drive time supplies 0.006 of River Bend's 0.004 lead over Rail Yard, so without it Rail Yard would be second.

## Limit. The package cannot say which weights are right

**Question.** Airport Road leads on the stated weights. Is it the answer?

**What is not known.** The weights are a judgment, and the package has no way to check one. `monte_carlo` draws 2,000 weight sets centred on the stated ones and spread nearly as wide as a uniform draw over all weights: a 0.30 weight is drawn with a standard deviation of about 0.2, and fewer than one draw in twenty lies within 0.10 of the stated weights on all four. The cell prints the spread. Under draws that wide the leader does not hold: Airport Road and River Bend each come first in about two draws in five, and 2,000 draws do not separate them. Near the stated weights it does hold. Airport Road leads while flooding carries more than 0.13 of the weight and cost carries between 0.13 and 0.41, and no move of 0.05 in one weight changes the leader.

**What would settle it.** The people who own the decision agreeing how much flooding counts. The computation has turned the open question into one they can answer: is flooding worth at least 0.13 of the decision, and cost between 0.13 and 0.41? The cell asserts today's state, so it fails on the day a change to the fixtures or the sampler makes the ranking robust, and this section is then rewritten.

In [ ]:
stated = np.array(list(WEIGHT.values()))
utilities = {r.alternative: np.array([r.raw_utilities[c] for c in WEIGHT]) for r in results}


def leader_under(weights: dict[str, float]) -> str:
    """Who leads when the same utilities are weighted differently."""
    return max(utilities, key=lambda name: sum(weights[c] * u for c, u in zip(WEIGHT, utilities[name])))


def with_weight(criterion: str, value: float) -> dict[str, float]:
    """One weight set to `value` and the other three rescaled in proportion, as `one_at_a_time` does."""
    return {c: value if c == criterion else w * (1 - value) / (1 - WEIGHT[criterion]) for c, w in WEIGHT.items()}


# What monte_carlo draws: a Dirichlet with alpha = weight x number of criteria. The cell repeats the draw and checks it
# against the package, so the spread printed below is the package's.
draws = np.random.default_rng(7).dirichlet(stated * len(stated), size=2000)
frequencies = monte_carlo(scorer, sites, n_samples=2000, seed=7)
first = (draws @ np.array([utilities[name] for name in sites]).T).argmax(axis=1)
assert all(abs(float((first == i).mean()) - frequencies[name]["1"]) < 1e-9 for i, name in enumerate(sites)), "this draw is monte_carlo's"

table(["criterion", "stated weight", "mean of draws", "sd of draws", "5th percentile", "95th percentile"],
      [[c, f"{w:.2f}", draws[:, i].mean(), draws[:, i].std(), np.percentile(draws[:, i], 5), np.percentile(draws[:, i], 95)]
       for i, (c, w) in enumerate(WEIGHT.items())])
near = float((abs(draws - stated) <= 0.10).all(axis=1).mean())
print(f"draws within 0.10 of the stated weights on all four: {near:.1%}")
assert all(0.17 < draws[:, i].std() < 0.22 for i in range(len(stated))), "a 0.30 weight is drawn with a standard deviation of about 0.2"
assert near < 0.05

blocks = {7: frequencies, 101: monte_carlo(scorer, sites, n_samples=2000, seed=101)}  # a second block of draws
print()
table(["site", "first", "second", "third", "fourth", "first, second block"],
      [[name, *(f"{frequencies[name][place]:.0%}" for place in "1234"), f"{blocks[101][name]['1']:.0%}"] for name in repo])
for block in blocks.values():
    assert not robustness_report(block).is_robust
    assert all(0.34 < block[site]["1"] < 0.44 for site in ("Airport Road", "River Bend")), "each is first in about two draws in five"
    assert abs(block["Airport Road"]["1"] - block["River Bend"]["1"]) < 0.08, "and the draws do not separate them"
    assert block["Old Mill"]["1"] == 0.0, "one thing no weights change: the dominated site never wins"

count = round(frequencies["Airport Road"]["1"] * 2000)
share = count / 2000
half = 1.96 * (share * (1 - share) / 2000) ** 0.5
print(f"\nAirport Road first in {count:,} of 2,000 draws ({share:.0%}; 95% interval {share - half:.0%} to {share + half:.0%})")
assert (count, f"{share - half:.0%}", f"{share + half:.0%}") == (764, "36%", "40%")
assert round(frequencies["River Bend"]["1"] * 2000) == 797

# Near the stated weights: where the leader changes when one weight moves and the others keep their proportions.
holds = {c: [x / 1000 for x in range(1001) if leader_under(with_weight(c, x / 1000)) == "Airport Road"] for c in WEIGHT}
print()
table(["criterion", "stated weight", "Airport Road leads from", "to"], [[c, f"{WEIGHT[c]:.2f}", min(v), max(v)] for c, v in holds.items()])
assert all(len(v) == round((max(v) - min(v)) * 1000) + 1 for v in holds.values()), "each is one interval"
assert (round(min(holds["flood_free_years"]), 2), round(min(holds["annual_cost"]), 2), round(max(holds["annual_cost"]), 2)) == (0.13, 0.13, 0.41)
assert all(leader_under(with_weight(c, WEIGHT[c] + step)) == "Airport Road" for c in WEIGHT for step in (-0.05, 0.05)), (
    "no move of 0.05 in one weight changes the leader"
)

## Confidence. What these numbers can bear

This section is about how far the numbers above can be trusted. They come from invented data, so it says nothing about any real output. Every number above was printed to three places. The section names the doubt at each level of the data and runs one doubt through to the report.

| Level | The doubt | Kind | Size | What the package does |
|---|---|---|---|---|
| **Arrives** | Each measurement is one number with no range. `flood_free_years` counts the years with no flood in a 30-year record, and a different 30 years would give a different count | measured over an assumed range; the size on real data is not known, because the fixture is invented and carries no error | taken below as 19 give or take 3 for River Bend, an assumption made for this cell | Takes each number as exact. A model file has no field for a range or a source |
| **Handled** | The weights are a judgment, not a measurement | estimated, from the draws of `monte_carlo`, whose weights spread over most of the simplex (a 0.30 weight has a standard deviation of about 0.2). A draw with another spread would prove the figure wrong | Airport Road first in 764 of 2,000 draws (38%, 95% interval 36% to 40%), River Bend in 797 (40%). Near the stated weights the leader holds: no move of 0.05 in one weight changes it | `one_at_a_time`, `monte_carlo` and `scenario_compare` vary the weights |
| **Handled** | The shape of each curve is a judgment | not known; running each curve at other parameters would settle it | no size | Nothing varies a curve's shape |
| **Reported** | A ranking reads as settled. Second and third place differ by 0.004 | measured | 0.696 against 0.692 | `robustness_report` says not robust. The ranking itself carries no sign of it |

The cell runs the first row's doubt to the report. River Bend's flood count moves across its assumed range; everything else stays fixed.

In [ ]:
doubt = range(16, 23)  # 19 give or take 3: an assumption for this cell, not a measurement
sweep = range(0, 31)  # the whole record, to find where the leader changes
place_at, total_at, leader_at = {}, {}, {}
for years in sweep:
    varied = {name: dict(raw) for name, raw in sites.items()}
    varied["River Bend"]["flood_free_years"] = years
    order = ranked(varied)
    place_at[years], total_at[years], leader_at[years] = list(order).index("River Bend") + 1, order["River Bend"], next(iter(order))

table(["flood-free years", "River Bend total", "River Bend place", "leader"],
      [[years, total_at[years], place_at[years], leader_at[years]] for years in range(16, 25)])

assert set(leader_at[years] for years in doubt) == {"Airport Road"}, "the leader survives this doubt"
assert set(place_at[years] for years in doubt) == {2, 3}, "second place does not"
assert place_at[18] == 3 and place_at[19] == 2, "one year of record separates second from third"
swing = max(total_at[years] for years in doubt) - min(total_at[years] for years in doubt)
lead = repo["Airport Road"] - repo["River Bend"]
assert round(swing, 3) == 0.050 and round(lead, 3) == 0.067 and swing < lead, "the doubt cannot move River Bend's total past the leader's"
assert swing > 10 * (repo["River Bend"] - repo["Rail Yard"]), "the doubt in one input is over ten times the margin the ranking reports"
flip = next(years for years in sweep if leader_at[years] != "Airport Road")
assert flip == 24 and all(leader_at[years] == "Airport Road" for years in range(flip)), "five more flood-free years give River Bend the lead"
assert all(place_at[years] == 2 for years in range(19, flip)) and place_at[flip] == 1

# the same doubt on all four sites at once: every record moved up to three years either way, capped at the 30 the record holds
ranges = [range(max(0, raw["flood_free_years"] - 3), min(30, raw["flood_free_years"] + 3) + 1) for raw in sites.values()]
together = [next(iter(ranked({name: dict(raw, flood_free_years=y) for (name, raw), y in zip(sites.items(), counts)})))
            for counts in itertools.product(*ranges)]
assert len(together) == 1372 and set(together) == {"Airport Road"}, "the leader holds in every combination"
assert round(repo["River Bend"], 3) == 0.696 and round(repo["Rail Yard"], 3) == 0.692 and round(repo["River Bend"] - repo["Rail Yard"], 3) == 0.004

Read together. Within three years either way of 19, who leads does not depend on River Bend's flood record: the doubt moves River Bend's total by at most 0.050 and its distance from the leader is 0.067. The leader holds at every count from 0 to 23 and changes at 24, five years above the stated count, so "does not depend" was checked over that range and holds no further. It also holds with all four sites' counts moved up to three years either way together, in each of 1,372 combinations.

Who comes second does depend on the record, and that is a property of this fixture. River Bend places second from 19 years up and third at 18, so 19 is the lowest count at which it places second, and any shortfall swaps second and third. Which direction of error costs more is the owners' call. A report that printed "1. Airport Road" and then "2 or 3: River Bend, Rail Yard" would say what the data can bear. The package's ranking prints 0.696 above 0.692.

The Limit above stands apart from this. It concerns the weights, and no measurement of the sites narrows it. The owners' judgment does.

## Your own model

Point the two paths at your own files and run the cell. It asks cases 2 and 4 of your data: does withdrawing any one alternative change the leader under the naive rule, and which weights are largely unspent? The naive rule is case 2's: each criterion scaled between the worst and best alternative present, with its direction read from the model's value function. The cell needs at least three alternatives and stops with a message under that. Every line it prints says how many alternatives, withdrawals or criteria it examined. It reads the two files and writes nothing.

It does not run the confidence section's doubt on your data: that needs a range for each measurement, and a model file has no field for one.

**Close without saving, or clear the output first.** The cell prints figures from your own decision.

In [ ]:
MY_MODEL = None  # for example "~/decisions/vendor_model.yaml"
MY_ALTERNATIVES = None  # for example "~/decisions/vendors.yaml"

if MY_MODEL and MY_ALTERNATIVES:
    my_scorer = MAUTScorer.from_yaml(Path(MY_MODEL).expanduser())
    mine = yaml.safe_load(Path(MY_ALTERNATIVES).expanduser().read_text())["alternatives"]
    if len(mine) < 3:
        raise ValueError("this cell withdraws one alternative and compares the leader of the rest: it needs at least three alternatives")
    with warnings.catch_warnings(record=True) as my_warnings:
        warnings.simplefilter("always")
        my_results = my_scorer.rank(mine)
    my_weight = {c.name: c.weight for c in my_scorer.criteria}
    # the model's direction, as the scorer has it: does utility fall between the lowest and the highest value present?
    falls = {c.name: c.value_fn(min(a[c.name] for a in mine.values())) > c.value_fn(max(a[c.name] for a in mine.values()))
             for c in my_scorer.criteria}

    def my_naive_leader(alternatives: dict) -> str:
        totals = dict.fromkeys(alternatives, 0.0)
        for c, weight in my_weight.items():
            low, high = min(a[c] for a in alternatives.values()), max(a[c] for a in alternatives.values())
            for name, raw in alternatives.items():
                scaled = (raw[c] - low) / ((high - low) or 1)
                totals[name] += weight * (1 - scaled if falls[c] else scaled)
        return max(totals, key=totals.get)

    my_naive = my_naive_leader(mine)
    withdrawals = [name for name in mine if name != my_naive]
    flips = [name for name in withdrawals if my_naive_leader({k: v for k, v in mine.items() if k != name}) != my_naive]
    print(f"{len(mine)} alternatives. leader, the package: {my_results[0].alternative}   leader, naive scaling: {my_naive}")
    print(f"{len(withdrawals)} withdrawals tried (each alternative but the naive leader); {len(flips)} change the naive leader: {flips or 'none'}")
    print(f"{len(my_weight)} criteria checked for a narrow range; {len(my_warnings)} largely unspent")
    for warning in my_warnings:
        print("largely unspent:", warning.message)
    my_passage = passage_chart(my_scorer, mine, title="My alternatives, from raw measurement to utility to total")

## Where next

- [The tour](tour_from_a_model_to_a_ranking.ipynb): the whole path once, with the three sensitivity tools the limit above draws on.
- `src/myproject/decision_science/value_functions.py`: the seven curves; case 3 is in `exponential`.
- `tests/unit/test_scorer.py`, `tests/unit/test_value_functions.py`: the tests each **Source** line names.
- `.claude/skills/writing-tours-and-case-notebooks/CASES.md`: the pattern this notebook follows, for writing one in another repo.